# Applied Machine Learning (CSAI2017P) — Lab Assignment 9
## Breast Cancer Diagnosis from Imaging Features

Experiments 12 · CO2, CO4 · **10 marks**

| | |
|---|---|
| Name | *Vivek Kuliyal* |
| SAP ID | *590028449* |
| Batch | *B5* |
| Due | announced in the lab |
| Lab quiz | LQ9, after submission |

**Datasets:** B3 (Breast Cancer Wisconsin Diagnostic)

---

### Before you start
- Attempt **2 of 3** in Section A, **1 of 2** in Section B. Section C is compulsory.
- Fit every transformer inside a `Pipeline`. Never fit on the test set.
- Set `random_state` wherever the question asks for reproducibility.
- Every question wants a short written observation, not just a number.
- Restart the kernel and run top-to-bottom before you submit.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

for m in (np, pd, sklearn):
    print(f"{m.__name__:12s} {m.__version__}")


numpy        2.3.5
pandas       2.3.3
sklearn      1.7.2


## Load the data

Replace with the loader for this assignment's anchor dataset — see `Anchor-Datasets.pdf` for the snippets.

In [5]:
# TODO: load the dataset for this assignment
from sklearn.datasets import load_breast_cancer
# Load dataset as a pandas DataFrame
data = load_breast_cancer(as_frame=True)

# Separate independent and dependent variables
X = data.data
y = data.target

# Create a complete DataFrame
df = data.frame

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nFeature Names:")
print(X.columns.tolist())

print("\nTarget Classes:")
print(data.target_names)

Dataset loaded successfully!
Dataset Shape: (569, 31)

First 5 rows:


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0



Feature Names:
['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension']

Target Classes:
['malignant' 'benign']


---

# Section A — attempt any 2 of 3 (2 marks each)

*attempt 2 of 3*

## A1. Load and inspect  &nbsp;&nbsp;`[2 marks]`

*Dataset: B3.*

a. Load the data, report the shape and the malignant/benign split.
b. Explain in two lines where the 30 features come from – they are not raw pixels.
c. Report the three features most correlated with the diagnosis.


In [6]:
# A1 — code

# A1 (a) - Load and inspect the dataset

from sklearn.datasets import load_breast_cancer
import pandas as pd

# Load dataset
data = load_breast_cancer(as_frame=True)

X = data.data
y = data.target

# Dataset shape
print("Dataset Shape:", X.shape)

# Count malignant and benign cases
print("\nDiagnosis Distribution:")
print(y.value_counts().rename(index={
    0: "Malignant",
    1: "Benign"
}))

# Percentage distribution
print("\nDiagnosis Percentage:")
print((y.value_counts(normalize=True) * 100).rename(index={
    0: "Malignant",
    1: "Benign"
}).round(2))


Dataset Shape: (569, 30)

Diagnosis Distribution:
target
Benign       357
Malignant    212
Name: count, dtype: int64

Diagnosis Percentage:
target
Benign       62.74
Malignant    37.26
Name: proportion, dtype: float64


**PART B**

The 30 features are not raw pixels. They are numerical measurements extracted from digitised images of fine-needle aspirates of breast masses.

These features describe characteristics of cell nuclei, such as:
- Radius
- Texture
- Perimeter
- Area
- Smoothness
- Compactness
- Concavity
- Symmetry
- Fractal dimension*

In [12]:

# A1 (c) - Correlation with diagnosis

# Combine features and target
df = X.copy()
df["diagnosis"] = y

# Calculate correlation with diagnosis
correlation = df.corr(numeric_only=True)["diagnosis"].drop("diagnosis")

# Find top 3 features by absolute correlation
top_3 = correlation.abs().sort_values(ascending=False).head(3)

print("Top 3 Most Correlated Features:")
print(correlation.loc[top_3.index])


Top 3 Most Correlated Features:
worst concave points   -0.793566
worst perimeter        -0.782914
mean concave points    -0.776614
Name: diagnosis, dtype: float64


**Observation (A1):**

> *The Breast Cancer Wisconsin dataset contains 569 records and 30 numerical input features. It includes 212 malignant cases and 357 benign cases. The features are extracted from digitised images of fine-needle aspirates and represent measurements of cell nuclei rather than raw pixels. The correlation analysis identifies the three features with the strongest relationship with the diagnosis, which may be useful for distinguishing malignant and benign tumours.*

## A2. First classifier  &nbsp;&nbsp;`[2 marks]`

*Dataset: B3.*

a. Fit logistic regression in a scaling pipeline with a stratified 80/20 split.
b. Report accuracy, precision, recall and F1 on the malignant class.
c. Report the number of malignant cases the model missed.


In [14]:
# A2 — code

# A2 - First Classifier using Logistic Regression

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Load dataset
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)

X = data.data
y = data.target

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=0,
    stratify=y
)

# Create pipeline
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=10000))
])

# Train model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=0)
recall = recall_score(y_test, y_pred, pos_label=0)
f1 = f1_score(y_test, y_pred, pos_label=0)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Number of malignant cases missed
missed_malignant = cm[0, 1]

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

print("\nLogistic Regression Results")
print("--------------------------------")
print("Accuracy:", round(accuracy, 4))
print("Precision (Malignant):", round(precision, 4))
print("Recall (Malignant):", round(recall, 4))
print("F1-Score (Malignant):", round(f1, 4))

print("\nConfusion Matrix:")
print(cm)

print("\nMalignant Cases Missed:", missed_malignant)

Training Samples: 455
Testing Samples: 114

Logistic Regression Results
--------------------------------
Accuracy: 0.9825
Precision (Malignant): 1.0
Recall (Malignant): 0.9524
F1-Score (Malignant): 0.9756

Confusion Matrix:
[[40  2]
 [ 0 72]]

Malignant Cases Missed: 2


**Observation (A2):**

> *The Logistic Regression model was trained using an 80:20 stratified train-test split with StandardScaler in a pipeline. Its performance was evaluated using accuracy, precision, recall and F1-score for the malignant class. The confusion matrix was used to identify incorrectly classified malignant cases. Missing malignant cases is particularly important because it may delay further medical investigation. Overall, the results help evaluate the model's ability to identify malignant tumours.*

## A3. Cross-validation  &nbsp;&nbsp;`[2 marks]`

*Dataset: B3.*

a. Run 10-fold stratified cross-validation and report mean and standard deviation of accuracy.
b. Compare with your single-split figure from A2.
c. State in two lines why the single split is unreliable on 569 rows.


In [ ]:
# A3 — code


**Observation (A3):**

> *write 2–6 lines here*

---

# Section B — attempt any 1 of 2 (4 marks each)

*attempt 1 of 2*

## B1. Optimising for recall  &nbsp;&nbsp;`[4 marks]`

*Dataset: B3.*

a. Train logistic regression, SVM and random forest, and report recall on the malignant class for each.
b. For your best model, lower the decision threshold until malignant recall reaches at least 0.99, and report the resulting precision and the number of false alarms.
c. In 6–8 lines, argue what recall a screening tool should target and what the false alarms cost.


In [15]:
# B1 — code

# B1 - Optimising for Recall

import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Load dataset
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=0,
    stratify=y
)

# Further split training data into training and validation
X_subtrain, X_val, y_subtrain, y_val = train_test_split(
    X_train, y_train,
    test_size=0.20,
    random_state=0,
    stratify=y_train
)

# Define three models
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=10000))
    ]),

    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(probability=True, random_state=0))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=0
    )
}

# Train and compare models using validation recall
results = {}
fitted_models = {}

for name, model in models.items():

    model.fit(X_subtrain, y_subtrain)

    val_pred = model.predict(X_val)

    recall = recall_score(
        y_val, val_pred,
        pos_label=0
    )

    results[name] = recall
    fitted_models[name] = model

    print(name, "Malignant Recall:", round(recall, 4))

# Display comparison
print("\nModel Comparison:")
print(pd.Series(results).sort_values(ascending=False))

# Select the model with highest validation recall
best_name = max(results, key=results.get)
best_model = fitted_models[best_name]

print("\nBest Model:", best_name)

# Obtain probability of malignant class (class 0)
malignant_index = list(best_model.classes_).index(0)

val_prob = best_model.predict_proba(X_val)[:, malignant_index]

# Find the highest threshold at or below 0.5
# that achieves at least 0.99 malignant recall
thresholds = np.sort(np.unique(val_prob))
thresholds = thresholds[thresholds <= 0.5]

best_threshold = None

for threshold in thresholds[::-1]:
    val_threshold_pred = np.where(val_prob >= threshold, 0, 1)

    if recall_score(
        y_val, val_threshold_pred,
        pos_label=0
    ) >= 0.99:
        best_threshold = threshold
        break

# Use the selected threshold on untouched test data
test_prob = best_model.predict_proba(X_test)[:, malignant_index]

if best_threshold is not None:

    test_pred = np.where(test_prob >= best_threshold, 0, 1)

    cm = confusion_matrix(y_test, test_pred, labels=[0, 1])

    print("\nThreshold Optimisation Results")
    print("--------------------------------")
    print("Best Model:", best_name)
    print("Selected Threshold:", round(best_threshold, 4))
    print("Test Accuracy:", round(accuracy_score(y_test, test_pred), 4))
    print("Test Precision (Malignant):",
          round(precision_score(y_test, test_pred, pos_label=0), 4))
    print("Test Recall (Malignant):",
          round(recall_score(y_test, test_pred, pos_label=0), 4))
    print("Test F1-Score (Malignant):",
          round(f1_score(y_test, test_pred, pos_label=0), 4))

    print("\nConfusion Matrix:")
    print(cm)

    print("False Alarms:", cm[1, 0])
    print("Missed Malignant Cases:", cm[0, 1])

else:
    print("No suitable threshold found.")


Logistic Regression Malignant Recall: 1.0
SVM Malignant Recall: 0.9706
Random Forest Malignant Recall: 0.9706

Model Comparison:
Logistic Regression    1.000000
SVM                    0.970588
Random Forest          0.970588
dtype: float64

Best Model: Logistic Regression

Threshold Optimisation Results
--------------------------------
Best Model: Logistic Regression
Selected Threshold: 0.2933
Test Accuracy: 0.9561
Test Precision (Malignant): 0.9302
Test Recall (Malignant): 0.9524
Test F1-Score (Malignant): 0.9412

Confusion Matrix:
[[40  2]
 [ 3 69]]
False Alarms: 3
Missed Malignant Cases: 2


**Observation (B1):**

> *write 2–6 lines here*

## B2. PCA and the cost of compression  &nbsp;&nbsp;`[4 marks]`

*Dataset: B3.*

a. Apply PCA inside the pipeline retaining 2, 5, 10 and 20 components.
b. Report cross-validated accuracy for each, plus the variance explained.
c. Plot the first two components coloured by diagnosis and explain in 4–6 lines what PCA bought and what it cost.


In [ ]:
# B2 — code


**Observation (B2):**

> *write 2–6 lines here*

---

# Section C — compulsory (2 marks)

*compulsory*

## C1. Would you deploy it  &nbsp;&nbsp;`[2 marks]`

*Dataset: B3.*

a. State the recall, precision and dataset size of your best model.
b. In 4–6 lines, list two things you would need to know about how the data was collected before letting this model near a patient.


In [ ]:
# C1 — code


**Observation (C1):**

> *write 2–6 lines here*

---

## Self-check before submitting

- [ ] Kernel restarted and run top-to-bottom, all outputs visible
- [ ] Every attempted question has a written observation
- [ ] No transformer fitted outside a pipeline
- [ ] Metrics reported with units where they have any
- [ ] File named `AML_LA09_<SAPID>.ipynb`
